In [1]:
import os
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import sys

project_root = Path(os.getcwd()).parent
print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.seed import set_seed
set_seed(42)

from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.train.trainer_transformer import CSTrainer
from src.transformer.network import CSTransformer


/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF


/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
EPOCHS = 150
BATCH_SIZE = 64
LR = 0.001
WEIGHT_DECAY = 0.05
EPOCHS = 200

TEST_INHIBITOR = "2-mercaptobenzimidazole"
NUM_CYCLE = [1, 2, 3, 4]
save_dir = project_root / "experiments" / "run_transformer"
SAVE_DIR = str(save_dir)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

[*] Device: cuda


In [ ]:
pipe = Pipeline(
    num_cycle=NUM_CYCLE,
    test_inhibitor=TEST_INHIBITOR,
    norm_feat=True,
    use_wavelet=False
)

train_dataset = CVADataset(
    vol=pipe.train_voltage,
    cur=pipe.train_current,
    desc_df=pipe.train_analyzed_data
)
val_dataset = CVADataset(
    vol=pipe.test_voltage,
    cur=pipe.test_current,
    desc_df=pipe.test_analyzed_data
)


In [4]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Size Train: {len(train_dataset)} samples")
print(f"Size Val: {len(val_dataset)} samples")

num_desc_features = train_dataset[0]["features"].shape[0]

Size Train: 2684 samples
Size Val: 776 samples


In [ ]:
model = CSTransformer(
    desc_dim=num_desc_features,   
    signal_length=968,
    d_model=256,                  
    n_head=4,
    d_inner=1024,
    num_decoder_blocks=6,
    num_cond_tokens=8,
    postnet_embedding_dim=512
)

In [ ]:

trainer = CSTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    save_dir=SAVE_DIR,
    vol_scaler=pipe.vol_scaler,
    cur_scaler=pipe.cur_scaler,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    area_loss_start_epoch=15,     
    area_loss_weight=0.001,
    use_weight_mask=True,
    peak_weight=5.0,
    peak_ranges=[(0, 200), (450, 650)]  
)

print("\n" + "="*40)
print("Start training Transformer")
print("="*40)
trainer.fit(epochs=EPOCHS)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



Start training Transformer
Training Transformer on cuda...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:05<00:00,  2.49it/s, val_loss=0.0239]


Epoch 1 | Train Loss: 3.2240 | Val Loss: 0.0228 | LR: 0.001000
Saved best model (Val Loss: 0.0228)


Epoch 2 [Val]: 100%|██████████| 13/13 [00:05<00:00,  2.53it/s, val_loss=0.1362]


...

Epoch 51 | Train Loss: 0.8432 | Val Loss: 0.1631 | LR: 0.000848


Epoch 52 [Val]: 100%|██████████| 13/13 [00:04<00:00,  2.61it/s, val_loss=0.1815]


Epoch 52 | Train Loss: 0.8099 | Val Loss: 0.1826 | LR: 0.000842


Epoch 53 [Train]:  41%|████▏     | 17/41 [03:03<04:19, 10.80s/it, loss=0.9426]


KeyboardInterrupt: 